
# Bài tập lớp và bài tập về nhà – Tìm kiếm đối kháng

- **Bài 1:** Minimax cho Tic-Tac-Toe 3x3
- **Bài 2:** Alpha-beta pruning cho Tic-Tac-Toe 3x3
- **Bài 3:** Minimax mở rộng cho 5x5, 10x10, NxN
- **Bài 4:** Alpha-beta mở rộng cho 5x5, 10x10, NxN
- **Bài 5:** Ứng dụng alpha-beta trong cờ tướng, cờ vua, cờ vây
- **Bài 6:** Giao diện đồ họa Tic-Tac-Toe bằng `ipywidgets`



In [ ]:

import math
from copy import deepcopy
from typing import List, Optional, Tuple

X = "X"
O = "O"
EMPTY = None


In [ ]:

def create_board(n: int) -> List[List[Optional[str]]]:
    return [[EMPTY for _ in range(n)] for _ in range(n)]

def next_player(board):
    x_count = sum(cell == X for row in board for cell in row)
    o_count = sum(cell == O for row in board for cell in row)
    return X if x_count == o_count else O

def actions(board):
    return [(i, j) for i in range(len(board)) for j in range(len(board)) if board[i][j] is EMPTY]

def result(board, action):
    i, j = action
    if board[i][j] is not EMPTY:
        raise ValueError("Ô đã được chọn.")
    new_board = deepcopy(board)
    new_board[i][j] = next_player(board)
    return new_board

def is_full(board):
    return all(cell is not EMPTY for row in board for cell in row)

def lines(board):
    n = len(board)
    out = []
    out.extend(board)
    out.extend([[board[i][j] for i in range(n)] for j in range(n)])
    out.append([board[i][i] for i in range(n)])
    out.append([board[i][n - 1 - i] for i in range(n)])
    return out

def winner(board):
    for line in lines(board):
        if line[0] is not EMPTY and all(cell == line[0] for cell in line):
            return line[0]
    return None

def terminal(board):
    return winner(board) is not None or is_full(board)

def utility(board):
    w = winner(board)
    if w == X:
        return 1
    if w == O:
        return -1
    return 0

def print_board(board):
    n = len(board)
    for i in range(n):
        row = [" " if c is None else c for c in board[i]]
        print(" | ".join(row))
        if i < n - 1:
            print("-" * (4 * n - 3))


## Bài 1 – Minimax cho Tic-Tac-Toe 3x3

In [ ]:

def minimax_3x3(board):

    if terminal(board):
        return None

    player = next_player(board)

    def max_value(state):
        if terminal(state):
            return utility(state)
        v = -math.inf
        for a in actions(state):
            v = max(v, min_value(result(state, a)))
        return v

    def min_value(state):
        if terminal(state):
            return utility(state)
        v = math.inf
        for a in actions(state):
            v = min(v, max_value(result(state, a)))
        return v

    best_action = None
    if player == X:
        best_score = -math.inf
        for a in actions(board):
            score = min_value(result(board, a))
            if score > best_score:
                best_score = score
                best_action = a
    else:
        best_score = math.inf
        for a in actions(board):
            score = max_value(result(board, a))
            if score < best_score:
                best_score = score
                best_action = a
    return best_action


In [ ]:

# Ví dụ nhanh cho Bài 1
board_3x3 = [
    [X, O, X],
    [O, X, None],
    [None, O, None],
]
print_board(board_3x3)
print("Nước đi tối ưu:", minimax_3x3(board_3x3))


## Bài 2 – Alpha-beta pruning cho Tic-Tac-Toe 3x3

In [ ]:

def alphabeta_3x3(board):

    if terminal(board):
        return None

    player = next_player(board)

    def max_value(state, alpha, beta):
        if terminal(state):
            return utility(state)
        v = -math.inf
        for a in actions(state):
            v = max(v, min_value(result(state, a), alpha, beta))
            alpha = max(alpha, v)
            if alpha >= beta:
                break
        return v

    def min_value(state, alpha, beta):
        if terminal(state):
            return utility(state)
        v = math.inf
        for a in actions(state):
            v = min(v, max_value(result(state, a), alpha, beta))
            beta = min(beta, v)
            if alpha >= beta:
                break
        return v

    best_action = None
    if player == X:
        best_score = -math.inf
        alpha, beta = -math.inf, math.inf
        for a in actions(board):
            score = min_value(result(board, a), alpha, beta)
            if score > best_score:
                best_score = score
                best_action = a
            alpha = max(alpha, best_score)
    else:
        best_score = math.inf
        alpha, beta = -math.inf, math.inf
        for a in actions(board):
            score = max_value(result(board, a), alpha, beta)
            if score < best_score:
                best_score = score
                best_action = a
            beta = min(beta, best_score)
    return best_action


In [ ]:

print("Minimax:", minimax_3x3(board_3x3))
print("Alpha-beta:", alphabeta_3x3(board_3x3))


## Bài 3 – Minimax cho 5x5, 10x10, NxN

In [ ]:

def heuristic(board, ai_player):

    opponent = O if ai_player == X else X
    score = 0
    n = len(board)

    all_lines = []
    all_lines.extend(board)
    all_lines.extend([[board[i][j] for i in range(n)] for j in range(n)])
    all_lines.append([board[i][i] for i in range(n)])
    all_lines.append([board[i][n - 1 - i] for i in range(n)])

    for line in all_lines:
        if ai_player in line and opponent in line:
            continue
        ai_count = sum(cell == ai_player for cell in line)
        op_count = sum(cell == opponent for cell in line)
        if op_count == 0:
            score += 10 ** ai_count
        elif ai_count == 0:
            score -= 10 ** op_count

    return score

def minimax_nxn(board, depth_limit=3, ai_player=X, depth=0):

    if terminal(board):
        return utility(board) * 10000

    if depth_limit is not None and depth >= depth_limit:
        return heuristic(board, ai_player)

    current = next_player(board)

    if current == ai_player:
        value = -math.inf
        for a in actions(board):
            value = max(value, minimax_nxn(result(board, a), depth_limit, ai_player, depth + 1))
        return value
    else:
        value = math.inf
        for a in actions(board):
            value = min(value, minimax_nxn(result(board, a), depth_limit, ai_player, depth + 1))
        return value

def best_move_minimax_nxn(board, depth_limit=3, ai_player=X):
    best_action = None
    best_score = -math.inf
    for a in actions(board):
        nb = result(board, a)
        score = minimax_nxn(nb, depth_limit=depth_limit, ai_player=ai_player, depth=1)
        if score > best_score:
            best_score = score
            best_action = a
    return best_action, best_score


In [ ]:

# Ví dụ cho 5x5
board_5x5 = create_board(5)
board_5x5[0][0] = X
board_5x5[1][1] = O
board_5x5[0][1] = X
board_5x5[2][2] = O
print_board(board_5x5)
move, score = best_move_minimax_nxn(board_5x5, depth_limit=3, ai_player=X)
print("Nước đi đề xuất:", move, "| Điểm:", score)


## Bài 4 – Alpha-beta cho 5x5, 10x10, NxN

In [ ]:

def alphabeta_nxn(board, depth_limit=3, ai_player=X, depth=0, alpha=-math.inf, beta=math.inf):
    if terminal(board):
        return utility(board) * 10000

    if depth_limit is not None and depth >= depth_limit:
        return heuristic(board, ai_player)

    current = next_player(board)

    if current == ai_player:
        value = -math.inf
        for a in actions(board):
            value = max(value, alphabeta_nxn(result(board, a), depth_limit, ai_player, depth + 1, alpha, beta))
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value
    else:
        value = math.inf
        for a in actions(board):
            value = min(value, alphabeta_nxn(result(board, a), depth_limit, ai_player, depth + 1, alpha, beta))
            beta = min(beta, value)
            if alpha >= beta:
                break
        return value

def best_move_alphabeta_nxn(board, depth_limit=3, ai_player=X):
    best_action = None
    best_score = -math.inf
    alpha, beta = -math.inf, math.inf
    for a in actions(board):
        nb = result(board, a)
        score = alphabeta_nxn(nb, depth_limit=depth_limit, ai_player=ai_player, depth=1, alpha=alpha, beta=beta)
        if score > best_score:
            best_score = score
            best_action = a
        alpha = max(alpha, best_score)
    return best_action, best_score


In [ ]:

move_ab, score_ab = best_move_alphabeta_nxn(board_5x5, depth_limit=3, ai_player=X)
print("Alpha-beta đề xuất:", move_ab, "| Điểm:", score_ab)



## Bài 5 – Ứng dụng alpha-beta trong cờ tướng, cờ vua, cờ vây

Trong các game có không gian trạng thái rất lớn như **cờ tướng**, **cờ vua** và **cờ vây**, alpha-beta được dùng để:

- cắt bỏ các nhánh chắc chắn không thể tốt hơn phương án hiện có;
- kết hợp với **hàm lượng giá** để đánh giá vị trí khi không thể duyệt tới cuối ván;
- đi kèm các kỹ thuật tăng tốc như:
  - sắp xếp nước đi tốt trước,
  - iterative deepening,
  - transposition table,
  - quiescence search,
  - selective extensions.

Trong thực tế:
- **cờ vua / cờ tướng:** alpha-beta là nền tảng của nhiều engine cổ điển;
- **cờ vây:** vì số nhánh cực lớn nên thường phải kết hợp thêm Monte Carlo Tree Search hoặc học sâu.


## Bài 6 – Giao diện đồ họa Tic-Tac-Toe trong notebook

In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output

class TicTacToeGUI:
    def __init__(self, n=3, ai_player=O, depth_limit=9):
        self.n = n
        self.board = create_board(n)
        self.human = X if ai_player == O else O
        self.ai = ai_player
        self.depth_limit = depth_limit
        self.buttons = [[widgets.Button(layout=widgets.Layout(width="60px", height="60px")) for _ in range(n)] for _ in range(n)]
        self.output = widgets.Output()
        self.status = widgets.HTML()
        self.reset_btn = widgets.Button(description="Reset", button_style="warning")
        self.reset_btn.on_click(self.reset)
        for i in range(n):
            for j in range(n):
                self.buttons[i][j].on_click(self._make_handler(i, j))
        self.render()

    def _make_handler(self, i, j):
        def handler(_):
            if terminal(self.board) or self.board[i][j] is not EMPTY:
                return
            if next_player(self.board) != self.human:
                return
            self.board[i][j] = self.human
            self.render()
            if not terminal(self.board):
                self.ai_move()
        return handler

    def ai_move(self):
        move, _ = best_move_alphabeta_nxn(self.board, depth_limit=self.depth_limit, ai_player=self.ai)
        if move is not None:
            i, j = move
            self.board[i][j] = self.ai
        self.render()

    def reset(self, _=None):
        self.board = create_board(self.n)
        self.render()

    def render(self):
        with self.output:
            clear_output(wait=True)
            print_board(self.board)

        for i in range(self.n):
            for j in range(self.n):
                val = self.board[i][j]
                self.buttons[i][j].description = " " if val is None else val
                self.buttons[i][j].disabled = terminal(self.board) or (val is not None)

        w = winner(self.board)
        if w is not None:
            self.status.value = f"<b>Kết quả:</b> {w} thắng."
        elif is_full(self.board):
            self.status.value = "<b>Kết quả:</b> Hòa."
        else:
            self.status.value = f"<b>Lượt đi:</b> {next_player(self.board)}"

    def display(self):
        grid = [widgets.HBox(self.buttons[i]) for i in range(self.n)]
        display(self.status, self.reset_btn, *grid, self.output)

game = TicTacToeGUI(n=3, ai_player=O, depth_limit=9)
game.display()
